In [ ]:
!pip -q install segmentation-models-pytorch

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import torch
import pandas as pd
import zipfile

# ====================================
# 1. Google Drive 경로
# ====================================

DRIVE_ROOT = Path("/content/drive/MyDrive/path-to-uc-foodwaste")

ZIP_PATH = DRIVE_ROOT / "dataset.zip"
SPLIT_CSV = DRIVE_ROOT / "data_split.csv"

# Colab 내부 저장 위치
EXTRACT_DIR = Path("/content")
LOCAL_DATASET = EXTRACT_DIR / "dataset"

# ====================================
# 2. GPU 확인
# ====================================

print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("런타임 설정에서 GPU를 선택하세요.")

# ====================================
# 3. ZIP 압축 해제
# ====================================

if not ZIP_PATH.exists():
    raise FileNotFoundError(ZIP_PATH)

if not SPLIT_CSV.exists():
    raise FileNotFoundError(SPLIT_CSV)

if not LOCAL_DATASET.exists():
    print("ZIP 압축 해제 중...")

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

    print("압축 해제 완료!")

# ====================================
# 4. 데이터 확인
# ====================================

df = pd.read_csv(SPLIT_CSV)

for split in ["train", "val", "test"]:

    image_dir = LOCAL_DATASET / split / "images"
    mask_dir = LOCAL_DATASET / split / "masks"

    images = {p.stem for p in image_dir.glob("*.jpg")}
    masks = {p.stem for p in mask_dir.glob("*.png")}

    csv_ids = set(
        df.loc[df["split"] == split, "filename"]
    )

    assert images == masks == csv_ids, (
        f"{split} 데이터 불일치!"
    )

    print(f"{split}: {len(images)}장")

print("\n데이터 준비 완료!")

In [ ]:

import random
import numpy as np
from PIL import Image, ImageOps

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF

HEIGHT = 512
WIDTH = 768
BATCH_SIZE = 4

class FoodWasteDataset(Dataset):

    def __init__(self, root, split, augment=False):
        self.image_dir = root / split / "images"
        self.mask_dir = root / split / "masks"
        self.augment = augment

        self.image_paths = sorted(
            self.image_dir.glob("*.jpg")
        )

        assert len(self.image_paths) > 0

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]
        mask_path = self.mask_dir / (
            image_path.stem + ".png"
        )

        with Image.open(image_path) as im:
            image = im.convert("RGB")

        with Image.open(mask_path) as mk:
            mask = mk.convert("L")

        image = image.resize(
            (WIDTH, HEIGHT),
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            (WIDTH, HEIGHT),
            Image.Resampling.NEAREST
        )

        # Train에서만 간단한 데이터 증강
        if self.augment and random.random() < 0.5:
            image = ImageOps.mirror(image)
            mask = ImageOps.mirror(mask)

        # RGB → Tensor + ImageNet 정규화
        image = TF.to_tensor(image)

        image = TF.normalize(
            image,
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        # Binary Mask → 0 또는 1
        mask = np.array(mask)
        mask = (mask > 0).astype(np.float32)

        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


train_dataset = FoodWasteDataset(
    LOCAL_DATASET, "train", augment=True
)

val_dataset = FoodWasteDataset(
    LOCAL_DATASET, "val", augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 데이터 형태 확인
images, masks = next(iter(train_loader))

print("Image shape:", images.shape)
print("Mask shape:", masks.shape)
print("Mask values:", torch.unique(masks))


In [ ]:

import segmentation_models_pytorch as smp
import torch.nn as nn

DEVICE = torch.device("cuda")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# U-Net + ResNet18
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

bce_loss = nn.BCEWithLogitsLoss()

def loss_fn(logits, targets):

    bce = bce_loss(logits, targets)

    probs = torch.sigmoid(logits)

    intersection = (
        probs * targets
    ).sum(dim=(1, 2, 3))

    denominator = (
        probs + targets
    ).sum(dim=(1, 2, 3))

    dice_loss = 1 - (
        (2 * intersection + 1) /
        (denominator + 1)
    ).mean()

    return bce + dice_loss


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("모델 생성 완료!")


In [ ]:

import torch
import pandas as pd

EPOCHS = 50
PATIENCE = 8

CHECKPOINT_PATH = DRIVE_ROOT / "unet_resnet18_best.pth"
HISTORY_PATH = DRIVE_ROOT / "training_history.csv"

scaler = torch.amp.GradScaler("cuda")
history = []

best_iou = -1
patience_count = 0

for epoch in range(EPOCHS):

    # ============================
    # Train
    # ============================

    model.train()
    train_loss_sum = 0

    for images, masks in train_loader:

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            logits = model(images)
            loss = loss_fn(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += loss.item() * images.size(0)

    train_loss = train_loss_sum / len(train_dataset)

    # ============================
    # Validation
    # ============================

    model.eval()

    val_loss_sum = 0
    TP, FP, FN = 0, 0, 0

    with torch.inference_mode():

        for images, masks in val_loader:

            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            with torch.amp.autocast("cuda"):
                logits = model(images)
                loss = loss_fn(logits, masks)

            val_loss_sum += loss.item() * images.size(0)

            probs = torch.sigmoid(logits.float())
            preds = probs > 0.5
            targets = masks > 0.5

            TP += (preds & targets).sum().item()
            FP += (preds & ~targets).sum().item()
            FN += (~preds & targets).sum().item()

    val_loss = val_loss_sum / len(val_dataset)

    iou = TP / (TP + FP + FN + 1e-7)
    dice = 2 * TP / (2 * TP + FP + FN + 1e-7)
    recall = TP / (TP + FN + 1e-7)
    precision = TP / (TP + FP + 1e-7)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_iou": iou,
        "val_dice": dice,
        "val_recall": recall,
        "val_precision": precision
    })

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"IoU: {iou:.4f} | "
        f"Dice: {dice:.4f} | "
        f"Recall: {recall:.4f}"
    )

    # ============================
    # Best Model 저장
    # ============================

    if iou > best_iou:

        best_iou = iou
        patience_count = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch + 1,
            "val_iou": best_iou,
            "height": HEIGHT,
            "width": WIDTH
        }, CHECKPOINT_PATH)

        print("  → Best Model 저장!")

    else:
        patience_count += 1

    # 학습 기록 저장
    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    # Early Stopping
    if patience_count >= PATIENCE:
        print("Early Stopping!")
        break

print("\n학습 종료")
print(f"최고 Validation IoU: {best_iou:.4f}")
print(f"모델 저장 위치: {CHECKPOINT_PATH}")


In [ ]:

from torch.utils.data import DataLoader
import pandas as pd
import torch

# 1. Test 데이터 준비
test_dataset = FoodWasteDataset(
    LOCAL_DATASET, "test", augment=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 2. 최고 성능 모델 불러오기
checkpoint = torch.load(
    DRIVE_ROOT / "unet_resnet18_best.pth",
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("불러온 Epoch:", checkpoint["epoch"])
print("저장된 Val IoU:", checkpoint["val_iou"])

# 3. Test 평가
metadata = pd.read_csv(SPLIT_CSV).set_index("filename")
records = []
cursor = 0

with torch.inference_mode():
    for images, masks in test_loader:

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.amp.autocast("cuda"):
            logits = model(images)

        preds = torch.sigmoid(logits.float()) >= 0.5
        targets = masks >= 0.5

        for j in range(images.size(0)):
            filename = test_dataset.image_paths[
                cursor + j
            ].stem

            pred = preds[j]
            target = targets[j]

            tp = (pred & target).sum().item()
            fp = (pred & ~target).sum().item()
            fn = (~pred & target).sum().item()

            records.append({
                "filename": filename,
                "group": metadata.loc[filename, "group"],
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "pred_area_pct": pred.float().mean().item() * 100,
                "true_area_pct": target.float().mean().item() * 100
            })

        cursor += images.size(0)

results = pd.DataFrame(records)

def calculate_metrics(data):
    tp = data["tp"].sum()
    fp = data["fp"].sum()
    fn = data["fn"].sum()

    return {
        "IoU": tp / (tp + fp + fn + 1e-7),
        "Dice": 2 * tp / (2 * tp + fp + fn + 1e-7),
        "Recall": tp / (tp + fn + 1e-7),
        "Precision": tp / (tp + fp + 1e-7)
    }

# 4. 전체 Test 성능
print("\n=== 전체 Test 성능 ===")
overall = calculate_metrics(results)

for name, value in overall.items():
    print(f"{name}: {value:.4f}")

# 5. A, C, D 그룹별 성능
print("\n=== 유형별 Test 성능 ===")

for group in ["A", "C", "D"]:
    subset = results[results["group"] == group]
    metrics = calculate_metrics(subset)

    print(f"\nGroup {group} ({len(subset)}장)")
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")

# 6. B그룹(정상 이미지) 오탐 분석
clean = results[results["group"] == "B"]

false_alarm_rate = (
    (clean["fp"] > 0).mean() * 100
)

avg_false_area = clean["pred_area_pct"].mean()

print("\n=== B그룹 정상 이미지 평가 ===")
print(f"정상 이미지: {len(clean)}장")
print(f"오탐 발생 이미지 비율: {false_alarm_rate:.2f}%")
print(f"평균 오탐 면적: {avg_false_area:.4f}%")

# 7. 상세 결과 저장
output = DRIVE_ROOT / "test_results_baseline.csv"

results.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\n결과 저장 완료:", output)


In [ ]:
import pandas as pd

df = pd.read_csv(
    DRIVE_ROOT / "test_results_baseline.csv"
)

# 이미지별 IoU, Recall 계산
positive = df[df["group"] != "B"].copy()

positive["iou"] = (
    positive["tp"] /
    (positive["tp"] + positive["fp"] + positive["fn"] + 1e-7)
)

positive["recall"] = (
    positive["tp"] /
    (positive["tp"] + positive["fn"] + 1e-7)
)

print("=== IoU가 가장 낮은 이미지 10장 ===")
print(
    positive.nsmallest(10, "iou")[
        ["filename", "group", "iou", "recall"]
    ].to_string(index=False)
)

print("\n=== 정상 이미지 오탐 면적 상위 10장 ===")

clean = df[df["group"] == "B"]

print(
    clean.nlargest(10, "pred_area_pct")[
        ["filename", "pred_area_pct"]
    ].to_string(index=False)
)

In [ ]:
import pandas as pd

split_df = pd.read_csv(SPLIT_CSV)

targets = [f"image{i:06d}" for i in range(290, 294)]

selected = split_df[
    split_df["filename"].isin(targets)
]

print("=== 290~293번 데이터 분할 ===")
print(
    selected[
        ["filename", "group", "block_id", "split", "area_ratio"]
    ].to_string(index=False)
)

# Test에 포함된 경우 오탐 결과 확인
test_results = pd.read_csv(
    DRIVE_ROOT / "test_results_baseline.csv"
)

special_test = test_results[
    test_results["filename"].isin(targets)
]

if not special_test.empty:
    print("\n=== Test 이미지 오탐 결과 ===")
    print(
        special_test[
            ["filename", "fp", "pred_area_pct"]
        ].to_string(index=False)
    )
else:
    print("\n290~293번은 Test에 포함되지 않았습니다.")

In [ ]:

from pathlib import Path
from PIL import Image
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch

# 결과 저장 폴더
SAVE_DIR = DRIVE_ROOT / "prediction_examples"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 기존 Test 평가 결과
results = pd.read_csv(
    DRIVE_ROOT / "test_results_baseline.csv"
)

# 이미지별 IoU 계산
results["iou"] = (
    results["tp"] /
    (results["tp"] + results["fp"] + results["fn"] + 1e-7)
)

# 확인할 이미지 선정
selected = []

# A: IoU가 낮은 4장
selected += results[
    results["group"] == "A"
].nsmallest(4, "iou")["filename"].tolist()

# B: 오탐 면적이 큰 4장
selected += results[
    results["group"] == "B"
].nlargest(4, "pred_area_pct")["filename"].tolist()

# C, D: IoU가 낮은 2장씩
for group in ["C", "D"]:
    selected += results[
        results["group"] == group
    ].nsmallest(2, "iou")["filename"].tolist()

# 참고: 바닥이 드러난 학습 이미지
selected += [
    f"image{i:06d}" for i in range(290, 294)
]

# 파일별 split 확인
split_df = pd.read_csv(SPLIT_CSV)
split_map = dict(
    zip(split_df["filename"], split_df["split"])
)

model.eval()

def save_prediction(filename):

    split = split_map[filename]

    img_path = (
        LOCAL_DATASET / split / "images" /
        f"{filename}.jpg"
    )

    mask_path = (
        LOCAL_DATASET / split / "masks" /
        f"{filename}.png"
    )

    # RGB 이미지
    image = Image.open(img_path).convert("RGB")
    image = image.resize(
        (WIDTH, HEIGHT),
        Image.Resampling.BILINEAR
    )

    rgb = np.array(image)

    # Ground Truth
    gt_image = Image.open(mask_path).convert("L")
    gt_image = gt_image.resize(
        (WIDTH, HEIGHT),
        Image.Resampling.NEAREST
    )

    gt = np.array(gt_image) > 0

    # 모델 입력
    x = TF.to_tensor(image)
    x = TF.normalize(
        x,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    x = x.unsqueeze(0).to(DEVICE)

    # 예측
    with torch.inference_mode():
        logits = model(x)
        probability = torch.sigmoid(logits.float())

    pred = probability[0, 0].cpu().numpy() >= 0.5

    # TP, FP, FN 영역
    tp = pred & gt
    fp = pred & ~gt
    fn = ~pred & gt

    # 오류 비교 이미지
    overlay = (rgb * 0.5).astype(np.uint8)

    overlay[tp] = [0, 200, 0]       # 초록: TP
    overlay[fp] = [255, 50, 50]     # 빨강: FP
    overlay[fn] = [50, 100, 255]    # 파랑: FN

    # 4개 이미지 비교
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("Original RGB")

    axes[0, 1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[0, 1].set_title("Ground Truth")

    axes[1, 0].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, 0].set_title("Prediction")

    axes[1, 1].imshow(overlay)
    axes[1, 1].set_title("TP=Green / FP=Red / FN=Blue")

    for ax in axes.flat:
        ax.axis("off")

    fig.suptitle(f"{filename} | {split}", fontsize=14)
    plt.tight_layout()

    output_path = SAVE_DIR / f"{filename}_comparison.png"
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"저장 완료: {filename} ({split})")


# 선정된 이미지 저장
for filename in selected:
    save_prediction(filename)

print(f"\n총 {len(selected)}개 비교 이미지 저장 완료")
print("저장 폴더:", SAVE_DIR)


In [ ]:

!pip -q install segmentation-models-pytorch

import io
import numpy as np
import torch
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

from pathlib import Path
from PIL import Image
from google.colab import drive, files
from torchvision.transforms import functional as TF

# ==========================================
# 1. 환경 및 모델 불러오기
# ==========================================

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/path-to-uc-foodwaste")
MODEL_PATH = DRIVE_ROOT / "unet_resnet18_best.pth"

SAVE_DIR = DRIVE_ROOT / "TU_predictions"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

HEIGHT = checkpoint.get("height", 512)
WIDTH = checkpoint.get("width", 768)

THRESHOLD = 0.5

print("모델 불러오기 완료!")
print("입력 크기:", WIDTH, "x", HEIGHT)

# ==========================================
# 2. TU 이미지 업로드
# ==========================================

print("\nTU 이미지 파일을 선택하세요.")
uploaded = files.upload()

# ==========================================
# 3. 이미지별 예측 및 결과 저장
# ==========================================

for filename, file_data in uploaded.items():

    if not filename.lower().endswith(
        (".jpg", ".jpeg", ".png", ".bmp")
    ):
        print("지원하지 않는 이미지 형식:", filename)
        continue

    # 원본 이미지
    image = Image.open(
        io.BytesIO(file_data)
    ).convert("RGB")

    # 학습할 때와 동일한 크기
    image = image.resize(
        (WIDTH, HEIGHT),
        Image.Resampling.BILINEAR
    )

    rgb = np.array(image)

    # ImageNet 정규화
    x = TF.to_tensor(image)
    x = TF.normalize(
        x,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    x = x.unsqueeze(0).to(DEVICE)

    # 모델 예측
    with torch.inference_mode():
        logits = model(x)
        prob = torch.sigmoid(logits.float())

    probability = prob[0, 0].cpu().numpy()
    pred = probability >= THRESHOLD

    # 원본 위에 예측 영역 표시
    overlay = rgb.copy()

    # 이물질 예측 영역을 빨간색으로 표시
    overlay[pred] = (
        0.5 * rgb[pred]
        + 0.5 * np.array([255, 0, 0])
    ).astype(np.uint8)

    # ======================================
    # 4. 결과 시각화
    # ======================================

    fig, axes = plt.subplots(
        2, 2, figsize=(12, 8)
    )

    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("Original RGB")

    axes[0, 1].imshow(
        pred, cmap="gray", vmin=0, vmax=1
    )
    axes[0, 1].set_title("Prediction Mask")

    axes[1, 0].imshow(overlay)
    axes[1, 0].set_title("Prediction Overlay (Red)")

    im = axes[1, 1].imshow(
        probability,
        cmap="viridis",
        vmin=0,
        vmax=1
    )
    axes[1, 1].set_title("Probability Map")
    fig.colorbar(im, ax=axes[1, 1])

    for ax in axes.flat:
        ax.axis("off")

    fig.suptitle(
        f"TU Prediction: {filename}",
        fontsize=14
    )

    plt.tight_layout()

    output_path = (
        SAVE_DIR / f"{Path(filename).stem}_prediction.png"
    )

    plt.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(
        f"{filename} | "
        f"예측 이물질 면적: {pred.mean()*100:.2f}%"
    )

print("\n모든 이미지 예측 완료!")
print("저장 위치:", SAVE_DIR)
